### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="habermans_survival",
    dataset_year="1970",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5XK51",
    download_description="""
We get the data from UCI.

wget https://archive.ics.uci.edu/static/public/43/haberman+s+survival.zip && unzip haberman+s+survival.zip && rm haberman+s+survival.zip haberman.names
mkdir -p local-data-warehouse/habermans_survival && mv haberman.data local-data-warehouse/habermans_survival/
""",
    # References
    academic_reference_bibtex="""@book{haberman1976generalized,
  title={Generalized residuals for log-linear models},
  author={Haberman, Shelby J},
  year={1976}
}
""",
    academic_reference_bibtex_key="haberman1976generalized",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Temporal"],
    curation_comments="""
We use the data without any further curation.

- Note, the data contains naturally occurring duplicates.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="survival_status",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    time_on="operation_year",
)

## Preprocessing

In [2]:
import pandas as pd

columns = [
    "age",
    "operation_year",
    "positive_axillary_nodes",
    "survival_status"
]

df = pd.read_csv(dataset_mold.path / "haberman.data", header=None, names=columns)
print("Loaded data shape:", df.shape)
df["survival_status"] = df["survival_status"].astype("category")
df["operation_year"] = pd.to_datetime(df["operation_year"] + 1900, format="%Y")

df = df.sort_values(by="operation_year").reset_index(drop=True)  # Shuffle the data

Loaded data shape: (306, 4)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 306
Columns: 4
Use sampling: False (sample size: 306)
Get row duplicates (staged, merged)...
Using top-3 columns for initial filtering: ['age', 'positive_axillary_nodes', 'operation_year']
Rows remaining as candidates after top-3 filter: 45 (of 306)

#### Duplicate Report
Total duplicate rows: 17 (5.56% of dataset)
Duplicate rows ignoring target: 23 (7.52% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,age,operation_year,positive_axillary_nodes,survival_status
0,83,1958-01-01,2,2
1,65,1958-01-01,0,2
2,64,1958-01-01,0,1
3,39,1958-01-01,0,1
4,40,1958-01-01,2,1


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,survival_status,category,0.0,0.0,2.0,"1, 2"
1,operation_year,datetime64[ns],0.0,0.0,12.0,"1958-01-01 00:00:00, 1964-01-01 00:00:00, 1963-01-01 00:00:00, 1960-01-01 00:00:00, 1965-01-01 00:00:00, 1966-01-01 00:00:00, 1959-01-01 00:00:00, 1961-01-01 00:00:00, 1967-01-01 00:00:00, 1962-01-01 00:00:00"
2,age,int64,0.0,0.0,49.0,"52, 54, 50, 47, 43, 53, 57, 41, 55, 38"
3,positive_axillary_nodes,int64,0.0,0.0,31.0,"0, 1, 2, 3, 4, 7, 8, 6, 9, 5"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,306.0,52.457516,10.803452,30.0,83.0
positive_axillary_nodes,306.0,4.026144,7.189654,0.0,52.0


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column          rank                                   
operation_year  1     1958-01-01 00:00:00     36  11.76
                2     1964-01-01 00:00:00     31  10.13
                3     1963-01-01 00:00:00     30   9.80
                4     1960-01-01 00:00:00     28   9.15
                5     1965-01-01 00:00:00     28   9.15
survival_status 1                       1    225  73.53
                2                       2     81  26.47

In [8]:
# Target Distribution
target_df

,count,pct
survival_status,,
1,225,73.53
2,81,26.47


## Task Curation

In [9]:
table = (
    df.groupby("operation_year")["survival_status"]
    .agg(count="size", survived_ratio=lambda s: (s == 1).mean())
    .reset_index()
)
table

,operation_year,count,survived_ratio
0,1958-01-01,36,0.666667
1,1959-01-01,27,0.666667
2,1960-01-01,28,0.857143
3,1961-01-01,26,0.884615
4,1962-01-01,23,0.695652
5,1963-01-01,30,0.733333
6,1964-01-01,31,0.741935
7,1965-01-01,28,0.535714
8,1966-01-01,28,0.785714
9,1967-01-01,25,0.840000


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata

years = sorted(df[task_mold.time_on].dropna().unique())
start_test_year = 3
splits = {}

for i in range(start_test_year, len(years)):
    test_year = years[i]
    train_years = years[:i]

    train_idx = list(df.index[df[task_mold.time_on].isin(train_years)])
    test_idx = list(df.index[df[task_mold.time_on].eq(test_year)])

    splits[i - start_test_year] = {
        0: (train_idx, test_idx)
    }
    print(len(train_idx), len(test_idx))

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We simulate a model that is refit every year. To have an initial start point, we always include the first 3 years as training data. We then create rolling window splits by always taking the next year as test data and all prior data as train data.",
    splits=splits,
)

91 26
117 23
140 30
170 31
201 28
229 28
257 25
282 13
295 11


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c7266-32de-7356-abc3-1c23864a89d5
3aff97a48f00d709f6d682fb0d937f9dc1ed631925f563b8dcf71170f2c5a9fe
